In [0]:
BASE_PATH = "/Volumes/customer360_dev/staging/source_files"

EXPECTED = {
    "customers": {
        "files": 10,
        "rows": 1_000_000,
        "key": "customer_id"
    },
    "addresses": {
        "files": 12,
        "rows": 1_200_000,
        "key": "address_id"
    },
    "subscriptions": {
        "files": 10,
        "rows": 1_000_000,
        "key": "subscription_id"
    },
    "products": {
        "files": 3,
        "rows": 150_000,
        "key": "product_id"
    },
    "orders": {
        "files": 20,
        "rows": 5_000_000,
        "key": "order_id"
    },
    "order_items": {
        "files": 30,
        "rows": 15_000_000,
        "key": "order_item_id"
    },
    "payments": {
        "files": 24,
        "rows": 6_000_000,
        "key": "payment_id"
    },
    "support_tickets": {
        "files": 15,
        "rows": 1_500_000,
        "key": "ticket_id"
    }
}

In [0]:
from pyspark.sql import functions as F

file_results = []

for entity, cfg in EXPECTED.items():
    path = f"{BASE_PATH}/{entity}/initial"
    
    files = dbutils.fs.ls(path)
    csv_files = [f for f in files if f.name.endswith(".csv")]
    
    actual_file_count = len(csv_files)
    total_size_bytes = sum(f.size for f in csv_files)
    
    file_results.append({
        "entity": entity,
        "expected_files": cfg["files"],
        "actual_files": actual_file_count,
        "file_count_status": "PASS" if actual_file_count == cfg["files"] else "FAIL",
        "total_size_mb": round(total_size_bytes / (1024 * 1024), 2)
    })

display(spark.createDataFrame(file_results))

actual_files,entity,expected_files,file_count_status,total_size_mb
10,customers,10,PASS,152.8
12,addresses,12,PASS,172.69
10,subscriptions,10,PASS,130.74
3,products,3,PASS,20.11
20,orders,20,PASS,836.29
30,order_items,30,PASS,1784.48
24,payments,24,PASS,940.09
15,support_tickets,15,PASS,301.99


### Validate total row counts

In [0]:
row_results = []

for entity, cfg in EXPECTED.items():
    path = f"{BASE_PATH}/{entity}/initial"

    df = (
        spark.read
        .option("header", True)
        .csv(path)
    )

    actual_rows = df.count()

    row_results.append({
        "entity": entity,
        "expected_rows": cfg["rows"],
        "actual_rows": actual_rows,
        "difference": actual_rows - cfg["rows"],
        "row_count_status": "PASS" if actual_rows == cfg["rows"] else "FAIL"
    })

display(spark.createDataFrame(row_results))

actual_rows,difference,entity,expected_rows,row_count_status
1000000,0,customers,1000000,PASS
1200078,78,addresses,1200000,PASS
1000066,66,subscriptions,1000000,PASS
150024,24,products,150000,PASS
5000126,126,orders,5000000,PASS
15000186,186,order_items,15000000,PASS
6000150,150,payments,6000000,PASS
1500096,96,support_tickets,1500000,PASS


### Validate missing business keys

In [0]:
key_results = []

for entity, cfg in EXPECTED.items():
    path = f"{BASE_PATH}/{entity}/initial"

    df = (
        spark.read
        .option("header", True)
        .csv(path)
    )

    key_col = cfg["key"]

    missing_keys = (
        df
        .filter(
            F.col(key_col).isNull() |
            (F.trim(F.col(key_col)) == "")
        )
        .count()
    )

    key_results.append({
        "entity": entity,
        "business_key": key_col,
        "missing_key_count": missing_keys
    })

display(spark.createDataFrame(key_results))

business_key,entity,missing_key_count
customer_id,customers,1000
address_id,addresses,0
subscription_id,subscriptions,0
product_id,products,0
order_id,orders,0
order_item_id,order_items,0
payment_id,payments,0
ticket_id,support_tickets,0


### Check duplicate business keys

In [0]:
duplicate_results = []

for entity, cfg in EXPECTED.items():
    path = f"{BASE_PATH}/{entity}/initial"

    df = (
        spark.read
        .option("header", True)
        .csv(path)
    )

    key_col = cfg["key"]

    duplicate_groups = (
        df
        .filter(F.col(key_col).isNotNull())
        .groupBy(key_col)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    duplicate_results.append({
        "entity": entity,
        "business_key": key_col,
        "duplicate_key_groups": duplicate_groups
    })

display(spark.createDataFrame(duplicate_results))

business_key,duplicate_key_groups,entity
customer_id,5000,customers
address_id,3,addresses
subscription_id,3,subscriptions
product_id,3,products
order_id,3,orders
order_item_id,3,order_items
payment_id,3,payments
ticket_id,3,support_tickets


### Validate schema columns

In [0]:
EXPECTED_COLUMNS = {
    "customers": [
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "phone_number",
        "date_of_birth",
        "gender",
        "signup_date",
        "customer_status",
        "loyalty_tier",
        "preferred_language",
        "created_at",
        "updated_at"
    ],

    "orders": [
        "order_id",
        "customer_id",
        "order_timestamp",
        "order_status",
        "order_channel",
        "currency_code",
        "gross_amount",
        "discount_amount",
        "tax_amount",
        "shipping_amount",
        "net_amount",
        "shipping_address_id",
        "billing_address_id",
        "promotion_code",
        "created_at",
        "updated_at"
    ]
}